# ML Notebook: Feast PostgreSQL Offline Store To Saved Model

This notebook uses Feast native APIs against the **core PostgreSQL offline store**:

1. Load generated ranking labels from PostgreSQL table `feature_store.ml_ranking_labels`.
2. Call native Feast `FeatureStore.get_historical_features(...)` against `bst_ranking_v1`; the feature views are `PostgreSQLSource` tables, not MinIO/S3 file sources.
3. Convert Feast historical features into BST JSONL rows and create train/validation/test splits.
4. Train the existing BST binary classification model.
5. Evaluate on validation data.
6. Save the model and Feast/PostgreSQL lineage as `.joblib`.

There are intentionally no shell/Kubernetes commands, no temporary Spark pod, and no notebook-side command helper. The local Python process only needs network access and credentials for the PostgreSQL offline store URI configured below.


## Setup: paths, PostgreSQL config, and imports


In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "apps/ml-system").exists():
            return candidate
    raise RuntimeError("Could not find RecSys-MLops repository root")


def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


ROOT = find_repo_root(Path.cwd().resolve())
ML_SYSTEM_SRC = ROOT / "apps/ml-system/src"
DATA_PLATFORM_SRC = ROOT / "apps/data-platform/src"
sys.path.insert(0, str(ML_SYSTEM_SRC))
sys.path.insert(0, str(DATA_PLATFORM_SRC))

FEAST_REPO_PATH = Path(os.getenv("FEAST_REPO_PATH", str(ROOT / "apps/data-platform/feature-store/feature_repo")))
FEAST_POSTGRES_HOST = os.getenv("FEAST_POSTGRES_HOST", "feature-postgres.recsys-dataflow.svc.cluster.local")
FEAST_POSTGRES_PORT = os.getenv("FEAST_POSTGRES_PORT", "5432")
FEAST_POSTGRES_DB = os.getenv("FEAST_POSTGRES_DB", "feature_store")
FEAST_POSTGRES_SCHEMA = os.getenv("FEAST_POSTGRES_SCHEMA", "feature_store")
FEAST_POSTGRES_USER = os.getenv("FEAST_POSTGRES_USER", "feast")
FEAST_POSTGRES_PASSWORD = os.getenv("FEAST_POSTGRES_PASSWORD", "feast")
FEAST_POSTGRES_SSLMODE = os.getenv("FEAST_POSTGRES_SSLMODE", "disable")
FEATURE_SERVICE_NAME = os.getenv("FEATURE_SERVICE_NAME", "bst_ranking_v1")
ENTITY_INPUT_PATH = os.getenv(
    "FEAST_ENTITY_INPUT_PATH",
    f"postgresql://{FEAST_POSTGRES_HOST}:{FEAST_POSTGRES_PORT}/{FEAST_POSTGRES_DB}/{FEAST_POSTGRES_SCHEMA}.ml_ranking_labels",
)
POSTGRES_LABEL_TABLE = ENTITY_INPUT_PATH.rsplit("/", 1)[-1]
MAX_HISTORY_LEN = int(os.getenv("NOTEBOOK_MAX_HISTORY_LEN", "5"))
LOCAL_FEAST_SAMPLE_ROWS = int(os.getenv("LOCAL_FEAST_SAMPLE_ROWS", "5000"))
RANDOM_SEED = int(os.getenv("NOTEBOOK_RANDOM_SEED", "42"))
TRAINING_PERCENT = float(os.getenv("NOTEBOOK_TRAINING_PERCENT", "1.0"))

BST_SPLIT_DIR = ROOT / "notebooks/data/feast_postgres_bst_split"
MODEL_DIR = ROOT / "notebooks/models"
CHECKPOINT_DIR = MODEL_DIR / "feast_postgres_bst_checkpoints"
MODEL_PATH = MODEL_DIR / "feast_postgres_bst_10epoch.joblib"
DATASET_METADATA_PATH = BST_SPLIT_DIR / "dataset_version_meta.json"
SPLIT_META_PATH = BST_SPLIT_DIR / "split_meta.json"

os.environ["FEAST_POSTGRES_HOST"] = FEAST_POSTGRES_HOST
os.environ["FEAST_POSTGRES_PORT"] = FEAST_POSTGRES_PORT
os.environ["FEAST_POSTGRES_DB"] = FEAST_POSTGRES_DB
os.environ["FEAST_POSTGRES_SCHEMA"] = FEAST_POSTGRES_SCHEMA
os.environ["FEAST_POSTGRES_USER"] = FEAST_POSTGRES_USER
os.environ["FEAST_POSTGRES_PASSWORD"] = FEAST_POSTGRES_PASSWORD
os.environ["FEAST_POSTGRES_SSLMODE"] = FEAST_POSTGRES_SSLMODE

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BST_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo root: {ROOT}")
print(f"Feast repo path: {FEAST_REPO_PATH.relative_to(ROOT)}")
print("Feast offline store: PostgreSQL")
print(f"PostgreSQL host: {FEAST_POSTGRES_HOST}:{FEAST_POSTGRES_PORT}")
print(f"PostgreSQL database/schema: {FEAST_POSTGRES_DB}/{FEAST_POSTGRES_SCHEMA}")
print(f"labels/entity table: {POSTGRES_LABEL_TABLE}")
print(f"feature service: {FEATURE_SERVICE_NAME}")


## Step 1. Load labels and Feast historical features from PostgreSQL

The feature repo uses `offline_store.type = postgres` and each FeatureView uses `PostgreSQLSource`. The notebook reads only generated labels as `entity_df`; all historical feature values are retrieved through native Feast point-in-time retrieval.


In [ ]:
from feast import FeatureStore

from cli.prepare_bst_training_data import (
    FEAST_FEATURE_REFS,
    _canonical_entity_frame,
    _feast_historical_to_bst_frame,
    _normalize_row,
    _read_postgres_table_as_pandas,
    _registry_path,
    _write_json,
    _write_jsonl,
)
from feature_store.feast_registry import apply_feature_repo
from lineage.dataset_versioning import processing_code_version as resolve_processing_code_version

apply_feature_repo(FEAST_REPO_PATH, skip_source_validation=True)
store = FeatureStore(repo_path=str(FEAST_REPO_PATH))
feature_service = store.get_feature_service(FEATURE_SERVICE_NAME)

labels_df = _read_postgres_table_as_pandas(ENTITY_INPUT_PATH)
labels_df = labels_df.sort_values("prediction_timestamp").head(LOCAL_FEAST_SAMPLE_ROWS)
if labels_df.empty:
    raise ValueError(f"No label/entity rows found in PostgreSQL table {POSTGRES_LABEL_TABLE}")

entity_df = _canonical_entity_frame(labels_df)

historical = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service,
    full_feature_names=True,
).to_df()

training_frame = _feast_historical_to_bst_frame(entity_df, historical, max_history_len=MAX_HISTORY_LEN)
if training_frame.empty:
    raise ValueError("No rows found after Feast PostgreSQL historical retrieval")

print(f"labels/entity rows: {len(entity_df):,}")
print(f"Feast historical rows: {len(historical):,}")
print(f"BST training rows: {len(training_frame):,}")
print(f"Feast feature refs: {len(FEAST_FEATURE_REFS)}")
print(training_frame.head(5)[["user_id", "target_item_id", "event_time", "label"]].to_string(index=False))


## Step 2. Create train/validation/test splits

In [ ]:
training_frame = training_frame.sort_values("prediction_timestamp")
rows = [_normalize_row(row, max_history_len=MAX_HISTORY_LEN) for _, row in training_frame.iterrows()]
train_end = int(len(rows) * 0.8)
val_end = train_end + int(len(rows) * 0.1)
splits = {
    "train": rows[:train_end],
    "val": rows[train_end:val_end],
    "test": rows[val_end:],
}

train_path = BST_SPLIT_DIR / "train.jsonl"
val_path = BST_SPLIT_DIR / "val.jsonl"
test_path = BST_SPLIT_DIR / "test.jsonl"
for split, split_rows in splits.items():
    _write_jsonl(split_rows, BST_SPLIT_DIR / f"{split}.jsonl")

processing_code = resolve_processing_code_version()
split_meta = {
    "feature_source": "feast",
    "offline_store_type": "postgresql",
    "retrieval_mode": "native_feast_feature_store_get_historical_features",
    "entity_input_path": ENTITY_INPUT_PATH,
    "postgres_label_table": POSTGRES_LABEL_TABLE,
    "postgres_host": FEAST_POSTGRES_HOST,
    "postgres_database": FEAST_POSTGRES_DB,
    "postgres_schema": FEAST_POSTGRES_SCHEMA,
    "feature_service_name": FEATURE_SERVICE_NAME,
    "feast_repo_path": str(FEAST_REPO_PATH.relative_to(ROOT)),
    "feast_registry_path": _registry_path(FEAST_REPO_PATH),
    "feast_features": FEAST_FEATURE_REFS,
    "processing_code_version": processing_code,
    "entity_rows": len(entity_df),
    "historical_rows": len(historical),
    "total_rows": len(rows),
    "train_rows": len(splits["train"]),
    "val_rows": len(splits["val"]),
    "test_rows": len(splits["test"]),
    "split_strategy": "temporal",
    "max_history_len": MAX_HISTORY_LEN,
}
dataset_metadata = {
    "dataset_run_id": "notebook-feast-postgres-bst",
    "feature_source": "feast",
    "offline_store_type": "postgresql",
    "entity_input_path": ENTITY_INPUT_PATH,
    "feature_service_name": FEATURE_SERVICE_NAME,
    "processing_code_version": processing_code,
    "split_counts": {split: len(split_rows) for split, split_rows in splits.items()},
    "schema_columns": sorted(training_frame.columns.tolist()),
}
_write_json(SPLIT_META_PATH, split_meta)
_write_json(DATASET_METADATA_PATH, dataset_metadata)

train_records = read_jsonl(train_path)
val_records = read_jsonl(val_path)
test_records = read_jsonl(test_path)
label_summary = {
    split: pd.Series([record["label"] for record in records]).value_counts().sort_index().to_dict()
    for split, records in {"train": train_records, "val": val_records, "test": test_records}.items()
}

print(f"train/val/test rows: {len(train_records):,} / {len(val_records):,} / {len(test_records):,}")
print("label distribution:", label_summary)
print("split metadata:", SPLIT_META_PATH.relative_to(ROOT))
print("sample BST JSONL record:")
print(json.dumps(train_records[0], indent=2)[:1200])


## Step 3. Configure BST model

In [ ]:
ITEM_NUM = 22_700
CATEGORY_NUM = 30
BRAND_NUM = 740
PRICE_BUCKET_NUM = 10
TIME_BUCKET_NUM = 9
EVENT_TYPE_NUM = 3

MODEL_CONFIG = {
    "model_args": {
        "n_heads": 2,
        "k_interests": 4,
        "embed_dim": 8,
        "seq_len": MAX_HISTORY_LEN + 1,
        "intermediate_size": 32,
        "hidden_dropout_prob": 0.1,
        "attn_dropout_prob": 0.1,
        "hidden_act": "relu",
        "layer_norm_eps": 1e-12,
        "padding_idx": 0,
        "reload_model": False,
        "save_path": str(CHECKPOINT_DIR),
        "item_num": ITEM_NUM,
        "category_num": CATEGORY_NUM,
        "brand_num": BRAND_NUM,
        "price_bucket_num": PRICE_BUCKET_NUM,
        "time_bucket_num": TIME_BUCKET_NUM,
        "event_type_num": EVENT_TYPE_NUM,
    },
    "data_args": {
        "num_workers": 0,
        "shuffle": True,
        "max_history_len": MAX_HISTORY_LEN,
        "train_data_path": str(train_path),
        "val_data_path": str(val_path),
        "test_data_path": str(test_path),
        "padding_idx": 0,
    },
    "training_args": {
        "num_epochs": 10,
        "learning_rate": 0.001,
        "weight_decay": 0.00001,
        "batch_size": 256,
        "num_workers": 0,
        "ranking_ks": [1, 3, 5, 10],
    },
}

print(f"train JSONL: {train_path.relative_to(ROOT)}")
print(f"validation JSONL: {val_path.relative_to(ROOT)}")
print(f"test JSONL: {test_path.relative_to(ROOT)}")
print(f"max history length: {MAX_HISTORY_LEN}")
print(f"training percent: {TRAINING_PERCENT}")

## Step 4. Train model

In [ ]:
import torch

from models.dataset import recommenderDataset
from models.trainer import Trainer


torch.manual_seed(RANDOM_SEED)
trainer = Trainer(MODEL_CONFIG)
train_dataset = recommenderDataset(MODEL_CONFIG["data_args"], split="train", percent=TRAINING_PERCENT)
val_dataset = recommenderDataset(MODEL_CONFIG["data_args"], split="val", percent=1.0)
train_loader = trainer.get_data_loader(train_dataset, shuffle=True)
val_loader = trainer.get_data_loader(val_dataset, shuffle=False)

print(f"dataset class: {train_dataset.__class__.__name__}")
print(f"model class: {trainer.model.__class__.__name__}")
print(f"trainer class: {trainer.__class__.__name__}")
print(f"training device: {trainer.device}")
print(f"epochs: {MODEL_CONFIG['training_args']['num_epochs']}")
print(f"training rows used: {len(train_dataset):,}")

history = []
for epoch in range(1, MODEL_CONFIG["training_args"]["num_epochs"] + 1):
    train_metrics = trainer.train(train_loader)
    val_metrics = trainer.evaluate(val_loader)
    trainer.scheduler.step(val_metrics["loss"])
    history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})
    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_metrics['loss']:.4f} train_auc={train_metrics['auc']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_auc={val_metrics['auc']:.4f} "
        f"val_ndcg@10={val_metrics.get('ndcg@10', 0.0):.4f}"
    )

model = trainer.model
metrics = {
    "epochs": MODEL_CONFIG["training_args"]["num_epochs"],
    "final_train": history[-1]["train"],
    "final_validation": history[-1]["val"],
    "history": history,
}

## Step 5. Evaluate on validation dataset

In [ ]:
validation_metrics = metrics["final_validation"]

print("validation metrics from apps/ml-system Trainer.evaluate:")
for key in sorted(validation_metrics):
    value = validation_metrics[key]
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

## Step 6. Save model as `.joblib`

In [ ]:
BST_MODEL_COLUMNS = [
    "user_id",
    "hist_item_id",
    "hist_event_type",
    "hist_category",
    "hist_brand",
    "hist_price_bucket",
    "hist_time",
    "target_item_id",
    "target_category",
    "target_brand",
    "target_price_bucket",
    "event_time",
    "label",
]

artifact = {
    "model_type": "BST",
    "dataset_class": "recommenderDataset",
    "trainer_class": "Trainer",
    "task": "binary_classification",
    "feature_source": "feast",
    "offline_store_type": "postgresql",
    "entity_input_path": ENTITY_INPUT_PATH,
    "postgres_label_table": POSTGRES_LABEL_TABLE,
    "postgres_host": FEAST_POSTGRES_HOST,
    "postgres_database": FEAST_POSTGRES_DB,
    "postgres_schema": FEAST_POSTGRES_SCHEMA,
    "feast_repo_path": str(FEAST_REPO_PATH.relative_to(ROOT)),
    "feature_service_name": FEATURE_SERVICE_NAME,
    "feature_columns": BST_MODEL_COLUMNS,
    "state_dict": {key: value.detach().cpu().numpy() for key, value in model.state_dict().items()},
    "model_config": MODEL_CONFIG,
    "metrics": metrics,
    "trained_device": str(trainer.device),
    "training_dataset_dir": str(BST_SPLIT_DIR.relative_to(ROOT)),
    "dataset_metadata": dataset_metadata,
    "split_metadata": split_meta,
    "train_jsonl": str(train_path.relative_to(ROOT)),
    "val_jsonl": str(val_path.relative_to(ROOT)),
    "test_jsonl": str(test_path.relative_to(ROOT)),
    "random_seed": RANDOM_SEED,
}
joblib.dump(artifact, MODEL_PATH)

print(f"saved model artifact: {MODEL_PATH.relative_to(ROOT)}")
print(f"artifact size bytes: {MODEL_PATH.stat().st_size:,}")
print("saved artifact keys:", sorted(artifact.keys()))


## Document main steps completed in this notebook

- **Step 1 - Load data through Feast PostgreSQL offline store:** Loaded generated labels from `feature_store.ml_ranking_labels`, then called native Feast `FeatureStore.get_historical_features(...)` using FeatureService `bst_ranking_v1` and PostgreSQL-backed FeatureViews.
- **Step 2 - Split data:** Converted Feast historical features to BST rows, then wrote temporal train, validation, and test JSONL splits.
- **Step 3 - Configure model:** Built the existing BST config against the generated split paths.
- **Step 4 - Train/evaluate:** Trained the existing BST binary classification model and evaluated validation metrics.
- **Step 5 - Save model:** Saved model weights, config, metrics, Feast repo path, PostgreSQL table lineage, and split metadata to `notebooks/models/feast_postgres_bst_10epoch.joblib`.

Proof expectation: notebook output should show `Feast offline store: PostgreSQL`, `FeatureStore.get_historical_features(...)` execution, non-zero historical rows, validation metrics, and saved `.joblib` artifact.
